In [1]:
import pandas as pd
import re
import numpy as np

## Model extraction procedure

In [2]:
questions = []
for test in ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']:
    test_questions = pd.read_csv(f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test}/questions.csv")
    test_questions['test_type'] = test
    questions.append(test_questions)
questions = pd.concat(questions)
questions['item_id'] = questions['question'] + questions['image_file']

In [3]:
def extract_numerical_responses(word):
    """Extract all numbers from the response."""
    # remove all commas for example 1,000 -> 1000
    num_answer = str(word).replace(",", "")

    # find all negative numbers and decimals 
    pattern = r'-?\d+(\.\d+)?'
    all_numbers = list(re.finditer(pattern, num_answer))
    all_numbers = [float(num.group()) for num in all_numbers if num != '']

    return all_numbers

def find_non_floats(r):
    response_word_count = len(str(r['agent_response']).split(" "))
    
    # if only 2 or fewer words appear in the string
    if response_word_count <= 2:
        res = str(r['agent_response'])
        res = res.replace(" ", "")
        res = res.replace(",", "")
        extracted_response = extract_numerical_responses(res)
        if extracted_response:
            return extracted_response[0]
        else:
            return np.nan
    else:
        return np.nan

def find_non_mc(r):
    item = questions[
        (questions["question"] == r["question"]) &
        (questions["image_file"] == r["image_file"])                
    ].iloc[0]
    response = str(r["agent_response"]).lower()
    mc1, mc2, mc3, mc4 = item["mc1"], item["mc2"], item["mc3"], item["mc4"]

    for mc in [mc4,mc3,mc2,mc1]:
        if mc != np.nan and response.find(str(mc).lower().strip()) != -1:
            return mc

    return np.nan

def extract_valid_response(r):
    test_type = r['test_type']
    if test_type in ['holf', 'holf2', 'chartqa-test-continuous', 'chartqa-test-continuous-human']:
        return find_non_floats(r)
    elif test_type in ["vlat", 'calvi-trick', 'calvi-standard']:
        return find_non_mc(r)
    elif test_type in ['ggr']:
        item = questions[
            (questions["question"] == r["question"]) &
            (questions["image_file"] == r["image_file"])                
        ].iloc[0]

        if item['text_input_type'] == 'open_ended_question':
            return find_non_floats(r)
        elif item['text_input_type'] == 'multiple_choice_question':
            return find_non_mc(r)
        else:
            return np.nan

def extract_valid_response(r):
    test_type = r['test_type']
    if test_type in ['holf', 'holf2', 'chartqa-test-continuous', 'chartqa-test-continuous-human']:
        return find_non_floats(r)
    elif test_type in ["vlat", 'calvi-trick', 'calvi-standard']:
        return find_non_mc(r)
    elif test_type in ['ggr']:
        item = questions[
            (questions["question"] == r["question"]) &
            (questions["image_file"] == r["image_file"])                
        ].iloc[0]

        if item['text_input_type'] == 'open_ended_question':
            return find_non_floats(r)
        elif item['text_input_type'] == 'multiple_choice_question':
            return find_non_mc(r)
        else:
            return np.nan

## Evaluate model responses 

In [4]:
def check_mc_answer(r):
    process_answer = lambda a : str(a).lower().strip()
    # try:
    #     if (r['test_type'] in ['ggr', 'vlat']) and r['agent_type'] == 'Human':
    #         return int(r['is_correct'])
    # except:
    #     print(r['test_type'], r['agent_type'])
    #     raise Error
    #     return np.nan

    # for ans
    # we do this because ggr already has floating point numerical values
    # since we convert them to strings, they don't work well
    if ((r['test_type'] in ['ggr', 'vlat']) and r['agent_type'] == 'Human'):
        try:
            return int(float(r['correct_answer']) == float(r['agent_response']))
        except:
            pass
        

    is_correct = process_answer(r['correct_answer']) == process_answer(r['agent_response'])

    # there may be multiple correct answers so check if this  is the case
    is_correct_2 = False
    is_correct_3 = False
    try:
        is_correct_2 = process_answer(r['correct_answer_2']) == process_answer(r['agent_response'])
        is_correct_3 = process_answer(r['correct_answer_3']) == process_answer(r['agent_response'])
    except:
        pass

    return int(is_correct or is_correct_2 or is_correct_3)

def find_correct_mc(model_responses_raw):
    """
    Find if the model response has the correct multiple choice answer
    """
    model_responses = model_responses_raw.copy()
    model_responses["is_correct"] = model_responses.apply(
        check_mc_answer, axis=1
    )

    return model_responses

In [50]:
# def get_relaxed_accuracy(agent_response, correct_answer, relax_accuracy_alpha):
#     actual = abs(agent_response)
#     correct = abs(correct_answer)
    
#     return int((actual - correct) <= (correct * relax_accuracy_alpha))

def get_relaxed_accuracy(agent_response, correct_answer, relax_accuracy_alpha):
    actual = agent_response
    correct = correct_answer
    relax = correct_answer * relax_accuracy_alpha
    upper_relax = correct_answer + relax
    lower_relax = correct_answer - relax

    if correct_answer < 0:
        return int(upper_relax <= actual <= lower_relax)
    
    return int(lower_relax <= actual <= upper_relax)

def find_relaxed_accuracy(model_responses_raw, test_name, relax_acc_alpha=0):
    """
    Find relaxed accuracy
    """
    model_responses = model_responses_raw.copy()
    model_responses["agent_response"] = model_responses["agent_response"].astype(float)  
    model_responses["is_correct"] = model_responses.apply(
        lambda r: get_relaxed_accuracy(r['agent_response'], r['correct_answer'], relax_acc_alpha), axis=1
    )

    return model_responses

In [51]:
def evaluate_model_response(
    model_responses, 
    test_type, 
    relax_acc_alpha=0
):
    if test_type in ['ggr', 'vlat', 'calvi-trick', 'calvi-standard']:
        model_responses = find_correct_mc(model_responses_raw=model_responses)
    else:
        model_responses = find_relaxed_accuracy(
            model_responses_raw=model_responses,
            test_name=test_type,
            relax_acc_alpha = relax_acc_alpha,
        )
    return model_responses

## Create Raw response dataframe

In [52]:
def create_raw_response_dataframe(model_configs):
    model_responses = []
    tests = ['ggr', 'vlat', 'calvi-standard', 'calvi-trick', 'holf', 'holf2', 'chartqa-test-continuous-human']
    aws_link = 'https://data-visualization-benchmark.s3.us-west-2.amazonaws.com'
    item_answers = questions[['item_id', 'correct_answer']]
    
    for model_config in model_configs:
        model = model_config['model']
        top_p_dir = model_config['top_p']
        temperature_dir = model_config['temp']
        
        _dir = f'responses/indist_instructions_question/{top_p_dir}/{temperature_dir}/{model.replace("/", "-")}'
        for test_type in tests:
            if model == 'Human':
                file_name = f'{aws_link}/{test_type}/{_dir}/processed_extracted_responses.csv'
                model_response_df = pd.read_csv(file_name)
            else:
                file_name = f'{aws_link}/{test_type}/{_dir}/model_responses.csv'
                model_response_df = pd.read_csv(file_name)
                model_response_df = model_response_df.rename(columns={
                    'agentType': 'agent_type'
                })

            model_response_df['test_type'] = test_type
            model_response_df['item_id'] = model_response_df['question'] + model_response_df['image_file']
            model_responses.append(model_response_df)
    
    model_responses = pd.concat(model_responses)
    model_responses['test_type'] = model_responses['test_type'].replace({
        'calvi-trick': 'calvi',
        'calvi-standard': 'calvi'
    })
    return model_responses

model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]
# raw_response_df = create_raw_response_dataframe(model_configs)

In [53]:
# model_responses

In [54]:
# a = raw_response_df.copy()
# a = a.dropna(subset=['agent_response'])
# a[
#     (a['agent_type'] == 'Human') 
#     & (a['test_type'] == 'vlat')
# ]['item_id'].nunique()

## Process and evaluate extracted model responses (relax $\alpha$ = 0.05)

In [55]:
# for only vlat and only the human responses, people respond as ‘$4.9’ instead of ‘$4.90’ 
# like in the other tests, so the answer is marked as incorrect
# use this dict to map to regular answers

human_vlat_map_dict = {
    '$5': '$5.00',
    '$12': '$12.00',
    '$4.9': '$4.90',
    '$15': '$15.00',
    '$16.7': '$16.70',
    '$17': '$17.00',
    '$23.4': '$23.40',
    '$5.1': '$5.10',
    '$45': '$45.00',
    '$35.4': '$35.40',
    '$5.2': '$5.20'
}

In [56]:
def create_evaluated_response_dataframe(model_configs, questions):
    model_responses = []
    tests = ['ggr', 'vlat', 'calvi-standard', 'calvi-trick', 'holf', 'holf2', 'chartqa-test-continuous-human']
    aws_link = 'https://data-visualization-benchmark.s3.us-west-2.amazonaws.com'
    item_answers = questions[['item_id', 'correct_answer']]
    
    for model_config in model_configs:
        model = model_config['model']
        top_p_dir = model_config['top_p']
        temperature_dir = model_config['temp']
        
        _dir = f'responses/indist_instructions_question/{top_p_dir}/{temperature_dir}/{model.replace("/", "-")}'
        for test_type in tests:        
            if model == 'Human':
                file_name = f'{aws_link}/{test_type}/{_dir}/processed_extracted_responses.csv'
                model_response_df = pd.read_csv(file_name)
                model_response_df['agent_response'] = model_response_df['agent_response'].apply(
                    lambda r : human_vlat_map_dict[r] if r in human_vlat_map_dict.keys() else r
                )
            else:
                file_name = f'{aws_link}/{test_type}/{_dir}/extracted_responses.csv'
                model_response_df = pd.read_csv(file_name)
    
            if 'agent_type' not in model_response_df.columns:
                model_response_df = model_response_df.rename(columns={
                    'agentType': 'agent_type'
                })

            relevant_columns = ['question', 'image_file', 'item_id', 'agent_type', 'agent_response']

            if 'is_correct' in model_response_df.columns:
                relevant_columns.append('is_correct')
    
            model_response_df['item_id'] = model_response_df['question'] + model_response_df['image_file']
            model_response_df = pd.merge(
                model_response_df[relevant_columns],
                item_answers,
                on='item_id'
            )
                
            model_response_df['test_type'] = test_type
            model_response_df['agent_response'] = model_response_df['agent_response'].astype(str)
            model_response_df['agent_response'] = model_response_df.apply(extract_valid_response, axis=1)
            model_response_df = evaluate_model_response(model_response_df, test_type, relax_acc_alpha=0.05)
            model_responses.append(model_response_df)
    
    model_responses = pd.concat(model_responses)
    model_responses['test_type'] = model_responses['test_type'].replace({
        'calvi-trick': 'calvi',
        'calvi-standard': 'calvi'
    })

    return model_responses

In [58]:
model_configs = [
    {'model': "Salesforce/blip2-flan-t5-xl", 'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    {'model': "Salesforce/blip2-flan-t5-xxl", 'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    {'model': "llava-hf/llava-1.5-7b-hf", 'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    {'model': "llava-hf/llava-1.5-13b-hf", 'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    {'model': "liuhaotian/llava-v1.6-34b", 'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    {'model': "google/pix2struct-chartqa-base", 'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    {'model': "google/matcha-chartqa", 'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    {'model': "GPT-4V", 'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    {'model': "Human", 'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'},
]

all_evaluated_responses_df = create_evaluated_response_dataframe(model_configs, questions)
all_evaluated_responses_df.to_csv("./dataframes/all_evaluated_responses_df.csv", index=False)

In [ ]:
# all_evaluated_responses_df.dropna(subset=['agent_response'])
all_evaluated_responses_df[
    (all_evaluated_responses_df['test_type'] == 'holf2') & 
    (all_evaluated_responses_df['agent_type'] == 'google/matcha-chartqa')
]


In [80]:
# a = all_evaluated_responses_df.dropna(subset=['agent_response'])
# items = a[
#     (a['agent_type'] == 'Human') 
#     & (a['test_type'] == 'vlat')
# ]['item_id']
# items.nunique()

### Sanity checks

## Relax accuracy dataframe